### Calculating Tilt for mooring applications

In [ ]:
# Calculating Tilt for mooring applications

In [ ]:
# ------------------------------------------------------------
# Imports

import pandas as pd
import numpy as np
import xarray as xr
import os
import plotly.express as px
import plotly.graph_objects as go
import matplotlib.pyplot as plt
from matplotlib.colors import LogNorm
import glob
import gsw
import datetime

# End imports
# ------------------------------------------------------------


In [ ]:
# ------------------------------------------------------------
# Definitions

def compute_orientation(df):
    """
    Calculate orientation angles from accelerometer and magnetometer data.
    Returns proper tilt angles in range -180° to +180° for mooring analysis.
    """
    Ax = df["Ax (g)"].to_numpy()
    Ay = df["Ay (g)"].to_numpy()
    Az = df["Az (g)"].to_numpy()
    Mx = df["Mx (mG)"].to_numpy()
    My = df["My (mG)"].to_numpy()
    Mz = df["Mz (mG)"].to_numpy()

    # Pitch and Roll from accelerometer
    pitch = np.arctan2(-Ax, np.sqrt(Ay**2 + Az**2))
    roll = np.arctan2(Ay, Az)

    # Yaw estimate from magnetometer and pitch/roll
    mag_x = Mx * np.cos(pitch) + Mz * np.sin(pitch)
    mag_y = Mx * np.sin(roll) * np.sin(pitch) + My * np.cos(roll) - Mz * np.sin(roll) * np.cos(pitch)
    yaw = np.arctan2(-mag_y, mag_x)

    # Convert radians to degrees - keep in proper -180 to +180 range for tilt analysis
    df["Pitch"] = np.degrees(pitch)
    df["Roll"] = np.degrees(roll) 
    df["Yaw"] = np.degrees(yaw)

    return df

# End Definitions
# ------------------------------------------------------------

In [ ]:
# ------------------------------------------------------------
# Start Main
# ------------------------------------------------------------

# Configuration parameters
inst_type = 'MAT1'
processing_ver = 2
fgen = '/datasets/work/oa-aapp-ocean/work/2023_12_FOCUS/mooring/proc_1/rec_202504'
# /datasets/work/oa-srsalt/work/preqa/SWOT/cal_val/jason_calval/all_mooring_data/data_in/rec_202508/BASJAS_PTSUV_202508

# Find all folders starting with MAT1
list_folders = sorted([d for d in os.listdir(fgen) 
                      if os.path.isdir(os.path.join(fgen, d)) and d.startswith('MAT1')])
print(f"Found {len(list_folders)} MAT1 folders: {list_folders}\n")

# Dictionary to store all dataframes from different instruments
all_instruments = {}

# Process each MAT1 folder
for f in list_folders:
    print(f"Processing folder: {f}")
    folder = os.path.join(fgen, f)
    os.chdir(folder)
    
    # Find *_AccellMag CSV files - try multiple search patterns
    # filist = sorted(glob.glob("*/*_AccelMag.csv"))  # One level deep
    filist = sorted(glob.glob("**/*_AccelMag.csv", recursive=True))  # Any depth
    
    file_in = filist[0]  # Process first file found
    print(f"  Loading: {file_in}")
    
    # Read accelerometer data with error handling for malformed lines
    try:
        df_temp = pd.read_csv(
            file_in, 
            parse_dates=["ISO 8601 Time"],
            on_bad_lines='warn',  # Warn about bad lines but skip them
            engine='python'        # More flexible parser
        )
    except Exception as e:
        print(f"  skipping...")
        # Fallback: skip bad lines entirely
        df_temp = pd.read_csv(
            file_in, 
            parse_dates=["ISO 8601 Time"],
            on_bad_lines='skip',   # Skip malformed lines
            engine='python'
        )
    
    df_temp.rename(columns={"ISO 8601 Time": "Time"}, inplace=True)
    
    # Store in dictionary with folder name as key
    all_instruments[f] = df_temp
    
    print(f"  Loaded {len(df_temp):,} rows")
    print(f"  Time range: {df_temp['Time'].min()} to {df_temp['Time'].max()}\n")

# Use the first instrument's data as the primary df for compatibility with existing code
df = list(all_instruments.values())[0]

print(f"Total instruments loaded: {len(all_instruments)}")
print("\nData preview from first instrument:")
print(df.head())

In [ ]:
# compute the orientations
df = compute_orientation(df)
print(f"Data shape after orientation calculation: {df.shape}")
print("Columns:", df.columns.tolist())
df.head()

In [ ]:
# ------------------------------------------------------------
# Time filtering setup
# ------------------------------------------------------------

# Get data time range
data_start = df['Time'].min()
data_end = df['Time'].max()
data_duration = (data_end - data_start).days

print(f" Available data time range:")
print(f"   Start: {data_start}")
print(f"   End: {data_end}")
print(f"   Duration: {data_duration} days")

# Use the full data range for plotting raw data
time_start = data_start
time_end = data_end

print(f"\nUsing full data range for analysis:")
print(f"   Start: {time_start}")
print(f"   End: {time_end}")
print(f"   Duration: {(time_end - time_start).days} days")

In [ ]:
# ------------------------------------------------------------
# Tilt Analysis
# ------------------------------------------------------------

# Apply time filtering to the data
df_accel_filtered = df[(df["Time"] >= time_start) & (df["Time"] <= time_end)]
print(f"Time filter applied: {time_start} to {time_end}")
print(f"Filtered accelerometer data: {len(df_accel_filtered):,} rows\n")

# Calculate overall mooring tilt angle (layover angle)
mooring_tilt_angle = np.sqrt(df_accel_filtered['Pitch']**2 + df_accel_filtered['Roll']**2)

# Calculate tilt direction (azimuth of tilt vector)
tilt_azimuth = np.degrees(np.arctan2(df_accel_filtered['Roll'], df_accel_filtered['Pitch']))

# Normalize to 0-360 degrees for compass direction
tilt_azimuth = (tilt_azimuth + 360) % 360

print(f"Mooring Tilt Statistics:")
print(f"  Mean tilt angle: {mooring_tilt_angle.mean():.3f}°")
print(f"  Standard deviation: {mooring_tilt_angle.std():.3f}°")
print(f"  Maximum tilt angle: {mooring_tilt_angle.max():.3f}°")

print(f"\nTilt Direction Statistics:")
print(f"  Mean tilt direction: {tilt_azimuth.mean():.1f}° (from North)")

print(f"\nPitch and Roll Statistics:")
print(f"  Pitch: {df_accel_filtered['Pitch'].mean():.3f}° ± {df_accel_filtered['Pitch'].std():.3f}°")
print(f"  Roll: {df_accel_filtered['Roll'].mean():.3f}° ± {df_accel_filtered['Roll'].std():.3f}°")
print(f"  Max Pitch: {df_accel_filtered['Pitch'].max():.3f}°")
print(f"  Max Roll: {df_accel_filtered['Roll'].max():.3f}°")


In [ ]:
# ------------------------------------------------------------
# Time Series Plot of Tilt Data
# ------------------------------------------------------------

print("Creating detailed time series plots of tilt data...\n")

fig, axes = plt.subplots(3, 1, figsize=(15, 12))

# 1. Individual pitch and roll components
axes[0].plot(df_accel_filtered['Time'], df_accel_filtered['Pitch'], 
             'b-', alpha=0.7, linewidth=0.8, label='Pitch')
axes[0].plot(df_accel_filtered['Time'], df_accel_filtered['Roll'], 
             'r-', alpha=0.7, linewidth=0.8, label='Roll')
axes[0].axhline(y=0, color='k', linestyle='-', alpha=0.3)
axes[0].set_title('Pitch and Roll Components Over Time')
axes[0].set_ylabel('Angle (°)')
axes[0].legend()
axes[0].grid(True, alpha=0.3)

# 2. Combined tilt magnitude
axes[1].plot(df_accel_filtered['Time'], mooring_tilt_angle, 
             'g-', alpha=0.8, linewidth=1.0)
axes[1].axhline(y=mooring_tilt_angle.mean(), color='red', linestyle='--', 
               label=f'Mean: {mooring_tilt_angle.mean():.3f}°', alpha=0.8)
axes[1].axhline(y=mooring_tilt_angle.quantile(0.95), color='orange', linestyle='--', 
               label=f'95th percentile: {mooring_tilt_angle.quantile(0.95):.3f}°', alpha=0.8)
axes[1].set_title('Combined Tilt Magnitude Over Time')
axes[1].set_ylabel('Tilt Angle (°)')
axes[1].legend()
axes[1].grid(True, alpha=0.3)

# 3. Tilt direction (azimuth)
axes[2].plot(df_accel_filtered['Time'], tilt_azimuth, 
             'purple', alpha=0.7, linewidth=0.8)
axes[2].axhline(y=tilt_azimuth.mean(), color='red', linestyle='--', 
               label=f'Mean direction: {tilt_azimuth.mean():.1f}°', alpha=0.8)
axes[2].set_title('Tilt Direction (Azimuth) Over Time')
axes[2].set_ylabel('Direction (° from North)')
axes[2].legend()
axes[2].grid(True, alpha=0.3)
axes[2].set_ylim(0, 360)

# # 4. Top instrument displacement
# top_inst_data = displacement_results['Top_inst']
# axes[3].plot(df_accel_filtered['Time'], top_inst_data['horizontal_displacement'], 
#              'orange', alpha=0.8, linewidth=1.0)
# axes[3].axhline(y=top_inst_data['horizontal_displacement'].mean(), color='red', linestyle='--', 
#                label=f'Mean: {top_inst_data["horizontal_displacement"].mean():.4f}m', alpha=0.8)
# axes[3].set_title('Top Instrument Horizontal Displacement Over Time')
# axes[3].set_ylabel('Displacement (m)')
# axes[3].set_xlabel('Time')
# axes[3].legend()
# axes[3].grid(True, alpha=0.3)

# Format x-axis for all subplots
for ax in axes:
    ax.tick_params(axis='x', rotation=45)

plt.tight_layout()
plt.show()

# Print some statistics about the time series
if len(df_accel_filtered) > 1:
    print(f"Time Series Statistics:")
    print(f"  Time span: {df_accel_filtered['Time'].min()} to {df_accel_filtered['Time'].max()}")
    print(f"  Duration: {(df_accel_filtered['Time'].max() - df_accel_filtered['Time'].min()).days} days")
    print(f"  Data points: {len(df_accel_filtered):,}")
    print(f"  Sampling interval: ~{(df_accel_filtered['Time'].iloc[1] - df_accel_filtered['Time'].iloc[0]).total_seconds():.0f} seconds")

    print(f"\nTilt Variability:")
    print(f"  Pitch range: {df_accel_filtered['Pitch'].min():.3f}° to {df_accel_filtered['Pitch'].max():.3f}°")
    print(f"  Roll range: {df_accel_filtered['Roll'].min():.3f}° to {df_accel_filtered['Roll'].max():.3f}°")
    print(f"  Tilt magnitude range: {mooring_tilt_angle.min():.3f}° to {mooring_tilt_angle.max():.3f}°")
    # print(f"  Max displacement range: {top_inst_data['horizontal_displacement'].min():.4f}m to {top_inst_data['horizontal_displacement'].max():.4f}m")
else:
    print(f"\n⚠ WARNING: No data available in the filtered time range!")
    print(f"  Filtered data has {len(df_accel_filtered)} rows")
    print(f"  Check that time_start and time_end overlap with available data")

In [ ]:
# # Timeseries plots
# print("Mooring velocity direction - 3D visualization")

# # Prepare data for 3D scatter plot with time animation
# # Using the actual mooring tilt data
# plot_data = df_accel_filtered[['Time', 'Pitch', 'Roll']].copy()

# # Calculate tilt magnitude for coloring
# plot_data['Tilt_Magnitude'] = np.sqrt(plot_data['Pitch']**2 + plot_data['Roll']**2)

# # Add a time grouping column for animation frames (e.g., by day or hour)
# # Using daily grouping for smoother animation
# plot_data['Day'] = plot_data['Time'].dt.date
# plot_data['TimeLabel'] = plot_data['Time'].dt.strftime('%Y-%m-%d %H:%M')

# # Create the 3D scatter plot with animation
# fig = px.scatter_3d(
#     plot_data, 
#     x='Pitch', 
#     y='Roll', 
#     z='Tilt_Magnitude', 
#     color='Tilt_Magnitude',          # Color points by tilt magnitude
#     animation_frame='Day',            # Animate by day
#     hover_data=['TimeLabel'],         # Show precise time on hover
#     range_x=[plot_data['Pitch'].min()-1, plot_data['Pitch'].max()+1],
#     range_y=[plot_data['Roll'].min()-1, plot_data['Roll'].max()+1],
#     range_z=[0, plot_data['Tilt_Magnitude'].max()+1],
#     title='Mooring Tilt: Pitch vs Roll vs Tilt Magnitude Over Time',
#     labels={
#         'Pitch': 'Pitch (°)',
#         'Roll': 'Roll (°)',
#         'Tilt_Magnitude': 'Tilt Magnitude (°)'
#     },
#     color_continuous_scale='Viridis'
# )

# # Adjust layout for better viewing
# fig.update_layout(
#     margin=dict(l=0, r=0, b=0, t=40),
#     scene=dict(
#         xaxis_title='Pitch (°)',
#         yaxis_title='Roll (°)',
#         zaxis_title='Tilt Magnitude (°)'
#     )
# )

# # Display the figure
# fig.show()

# print(f"\nPlot contains {len(plot_data):,} data points")
# print(f"Animation frames: {plot_data['Day'].nunique()} days")


In [ ]:
# ------------------------------------------------------------
# Raw Sensor Data Time Series Plots - Multi-Instrument
# ------------------------------------------------------------

print(f"Creating raw sensor data plots for {len(all_instruments)} instruments...\n")
print("NOTE: Plots show INDIVIDUAL instrument data (not averaged)\n")

# Define color palette for different instruments
colors = plt.cm.tab10(np.linspace(0, 1, len(all_instruments)))

fig, axes = plt.subplots(2, 1, figsize=(15, 10))

# Process each instrument
for idx, (inst_name, df_inst) in enumerate(all_instruments.items()):
    # Apply time filtering
    df_inst_filtered = df_inst[(df_inst["Time"] >= time_start) & (df_inst["Time"] <= time_end)]
    
    color = colors[idx]
    
    # Plot 1: Accelerometer components for this instrument (using different line styles per axis)
    axes[0].plot(df_inst_filtered['Time'], df_inst_filtered['Ax (g)'], 
                alpha=0.5, linewidth=0.7, label=f'{inst_name} - Ax', color=color, linestyle='-')
    axes[0].plot(df_inst_filtered['Time'], df_inst_filtered['Ay (g)'], 
                alpha=0.5, linewidth=0.7, label=f'{inst_name} - Ay', color=color, linestyle='--')
    axes[0].plot(df_inst_filtered['Time'], df_inst_filtered['Az (g)'], 
                alpha=0.5, linewidth=0.7, label=f'{inst_name} - Az', color=color, linestyle=':')
    
    # Plot 2: Magnetometer components for this instrument
    axes[1].plot(df_inst_filtered['Time'], df_inst_filtered['Mx (mG)'], 
                alpha=0.5, linewidth=0.7, label=f'{inst_name} - Mx', color=color, linestyle='-')
    axes[1].plot(df_inst_filtered['Time'], df_inst_filtered['My (mG)'], 
                alpha=0.5, linewidth=0.7, label=f'{inst_name} - My', color=color, linestyle='--')
    axes[1].plot(df_inst_filtered['Time'], df_inst_filtered['Mz (mG)'], 
                alpha=0.5, linewidth=0.7, label=f'{inst_name} - Mz', color=color, linestyle=':')
    
    print(f"{inst_name} Raw Sensor Statistics (Individual - Not Averaged):")
    print(f"  Ax: {df_inst_filtered['Ax (g)'].mean():.4f}g ± {df_inst_filtered['Ax (g)'].std():.4f}g")
    print(f"  Ay: {df_inst_filtered['Ay (g)'].mean():.4f}g ± {df_inst_filtered['Ay (g)'].std():.4f}g")
    print(f"  Az: {df_inst_filtered['Az (g)'].mean():.4f}g ± {df_inst_filtered['Az (g)'].std():.4f}g")
    print(f"  Mx: {df_inst_filtered['Mx (mG)'].mean():.2f}mG ± {df_inst_filtered['Mx (mG)'].std():.2f}mG")
    print(f"  My: {df_inst_filtered['My (mG)'].mean():.2f}mG ± {df_inst_filtered['My (mG)'].std():.2f}mG")
    print(f"  Mz: {df_inst_filtered['Mz (mG)'].mean():.2f}mG ± {df_inst_filtered['Mz (mG)'].std():.2f}mG")
    print(f"  Data points: {len(df_inst_filtered):,}\n")

# Configure plot 1 - Accelerometer
axes[0].axhline(y=0, color='k', linestyle='-', alpha=0.3)
axes[0].set_title('Accelerometer Data Over Time - All Instruments (Individual Components - Not Averaged)', 
                  fontsize=12, fontweight='bold')
axes[0].set_ylabel('Acceleration (g)')
axes[0].legend(loc='best', fontsize=8, ncol=2)
axes[0].grid(True, alpha=0.3)

# Configure plot 2 - Magnetometer
axes[1].axhline(y=0, color='k', linestyle='-', alpha=0.3)
axes[1].set_title('Magnetometer Data Over Time - All Instruments (Individual Components - Not Averaged)', 
                  fontsize=12, fontweight='bold')
axes[1].set_ylabel('Magnetic Field (mG)')
axes[1].set_xlabel('Time')
axes[1].legend(loc='best', fontsize=8, ncol=2)
axes[1].grid(True, alpha=0.3)

# Format x-axis for all subplots
for ax in axes:
    ax.tick_params(axis='x', rotation=45)

plt.tight_layout()
plt.show()

print("\n" + "="*60)
print("Multi-instrument raw sensor comparison complete")
print("All data shown is INDIVIDUAL per instrument (not averaged)")
print("="*60)

In [ ]:
# ------------------------------------------------------------
# Multi-Instrument Overlay Plots
# ------------------------------------------------------------

print(f"Creating overlay plots for {len(all_instruments)} instruments...\n")

# Define color palette for different instruments
colors = plt.cm.tab10(np.linspace(0, 1, len(all_instruments)))

fig, axes = plt.subplots(3, 1, figsize=(15, 14))

# Process each instrument and compute orientations
for idx, (inst_name, df_inst) in enumerate(all_instruments.items()):
    # Apply time filtering
    df_inst_filtered = df_inst[(df_inst["Time"] >= time_start) & (df_inst["Time"] <= time_end)]
    
    # Compute orientation for this instrument
    df_inst_filtered = compute_orientation(df_inst_filtered.copy())
    
    # Calculate mooring tilt angle
    tilt_angle = np.sqrt(df_inst_filtered['Pitch']**2 + df_inst_filtered['Roll']**2)
    
    color = colors[idx]
    
    # Plot 1: Pitch comparison
    axes[0].plot(df_inst_filtered['Time'], df_inst_filtered['Pitch'], 
                alpha=0.6, linewidth=0.8, label=inst_name, color=color)
    
    # Plot 2: Roll comparison
    axes[1].plot(df_inst_filtered['Time'], df_inst_filtered['Roll'], 
                alpha=0.6, linewidth=0.8, label=inst_name, color=color)
    
    # Plot 3: Combined tilt magnitude
    axes[2].plot(df_inst_filtered['Time'], tilt_angle, 
                alpha=0.6, linewidth=0.8, label=inst_name, color=color)
    
    print(f"{inst_name}:")
    print(f"  Mean Pitch: {df_inst_filtered['Pitch'].mean():.3f}° ± {df_inst_filtered['Pitch'].std():.3f}°")
    print(f"  Mean Roll: {df_inst_filtered['Roll'].mean():.3f}° ± {df_inst_filtered['Roll'].std():.3f}°")
    print(f"  Mean Tilt: {tilt_angle.mean():.3f}° ± {tilt_angle.std():.3f}°")
    print(f"  Data points: {len(df_inst_filtered):,}\n")

# Configure plot 1 - Pitch
axes[0].axhline(y=0, color='k', linestyle='-', alpha=0.3)
axes[0].set_title('Pitch Angle Comparison - All Instruments', fontsize=12, fontweight='bold')
axes[0].set_ylabel('Pitch (°)')
axes[0].legend(loc='best', fontsize=9)
axes[0].grid(True, alpha=0.3)

# Configure plot 2 - Roll
axes[1].axhline(y=0, color='k', linestyle='-', alpha=0.3)
axes[1].set_title('Roll Angle Comparison - All Instruments', fontsize=12, fontweight='bold')
axes[1].set_ylabel('Roll (°)')
axes[1].legend(loc='best', fontsize=9)
axes[1].grid(True, alpha=0.3)

# Configure plot 3 - Tilt Magnitude
axes[2].set_title('Tilt Magnitude Comparison - All Instruments', fontsize=12, fontweight='bold')
axes[2].set_ylabel('Tilt Angle (°)')
axes[2].set_xlabel('Time')
axes[2].legend(loc='best', fontsize=9)
axes[2].grid(True, alpha=0.3)

# Format x-axis for all subplots
for ax in axes:
    ax.tick_params(axis='x', rotation=45)

plt.tight_layout()
plt.show()

print("\n" + "="*60)
print("Multi-instrument comparison complete")
print("="*60)

In [ ]:
# ------------------------------------------------------------
# Multi-Instrument Raw Sensor Overlay Plots
# ------------------------------------------------------------

print(f"Creating raw sensor overlay plots for {len(all_instruments)} instruments...\n")

# Define color palette for different instruments
colors = plt.cm.tab10(np.linspace(0, 1, len(all_instruments)))

fig, axes = plt.subplots(2, 1, figsize=(15, 12))

# Process each instrument
for idx, (inst_name, df_inst) in enumerate(all_instruments.items()):
    # Apply time filtering
    df_inst_filtered = df_inst[(df_inst["Time"] >= time_start) & (df_inst["Time"] <= time_end)]
    
    # Compute orientation for this instrument
    df_inst_filtered = compute_orientation(df_inst_filtered.copy())
    
    color = colors[idx]
    
    # Plot 1: Accelerometer comparison - using combined magnitude
    accel_mag = np.sqrt(df_inst_filtered['Ax (g)']**2 + df_inst_filtered['Ay (g)']**2 + df_inst_filtered['Az (g)']**2)
    axes[0].plot(df_inst_filtered['Time'], accel_mag, 
                alpha=0.6, linewidth=0.8, label=inst_name, color=color)
    
    # Plot 2: Magnetometer comparison - using combined magnitude
    mag_mag = np.sqrt(df_inst_filtered['Mx (mG)']**2 + df_inst_filtered['My (mG)']**2 + df_inst_filtered['Mz (mG)']**2)
    axes[1].plot(df_inst_filtered['Time'], mag_mag, 
                alpha=0.6, linewidth=0.8, label=inst_name, color=color)
    
    print(f"{inst_name}:")
    print(f"  Mean Accel Magnitude: {accel_mag.mean():.4f}g ± {accel_mag.std():.4f}g")
    print(f"  Mean Mag Magnitude: {mag_mag.mean():.2f}mG ± {mag_mag.std():.2f}mG")
    print(f"  Data points: {len(df_inst_filtered):,}\n")

# Configure plot 1 - Accelerometer
axes[0].set_title('Accelerometer Magnitude Comparison - All Instruments', fontsize=12, fontweight='bold')
axes[0].set_ylabel('Acceleration Magnitude (g)')
axes[0].legend(loc='best', fontsize=9)
axes[0].grid(True, alpha=0.3)

# Configure plot 2 - Magnetometer
axes[1].set_title('Magnetometer Magnitude Comparison - All Instruments', fontsize=12, fontweight='bold')
axes[1].set_ylabel('Magnetic Field Magnitude (mG)')
axes[1].set_xlabel('Time')
axes[1].legend(loc='best', fontsize=9)
axes[1].grid(True, alpha=0.3)

# Format x-axis for all subplots
for ax in axes:
    ax.tick_params(axis='x', rotation=45)

plt.tight_layout()
plt.show()

print("\n" + "="*60)
print("Multi-instrument raw sensor comparison complete")
print("="*60)